## Scikit Learn Pipelines

# Why Do We Need `sklearn` Pipelines?

## ✅ What is a Pipeline?
A Pipeline in `scikit-learn` helps you chain together multiple steps (like preprocessing + model) so that you can treat them as a single object.

---

## 🚀 Why Use Pipelines?

- **Clean & Organized Code**  
  Combines preprocessing and modeling into one flow.

- **No Data Leakage**  
  Ensures that data transformation (like scaling) is only done on training data during model fitting.

- **Easier Cross-Validation**  
  Automatically applies all steps correctly in each fold.

- **Reusability**  
  Easy to reuse the same steps on new data (like test data).

- **Fewer Errors**  
  Less chance of skipping a step or applying transformations wrongly.

---


![sklearn_pipelines](./sklearn_pipelines.png)

lets work on titanic dataset without using pipelines

In [3]:
#imporing libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier

In [4]:
df = pd.read_csv('train.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'],inplace = True) #dropping unimportant columns
df

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S
...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S
887,1,1,female,19.0,0,0,30.0000,S
888,0,3,female,NaN,1,2,23.4500,S
889,1,1,male,26.0,0,0,30.0000,C


In [6]:
# step 1 --> Train test split
#Survied is our target column and rest are feature colum
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['Survived']),df['Survived'],test_size=0.2,random_state=42)

In [7]:
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5000,S
733,2,male,23.0,0,0,13.0000,S
382,3,male,32.0,0,0,7.9250,S
704,3,male,26.0,1,0,7.8542,S
813,3,female,6.0,4,2,31.2750,S


In [11]:
X_test.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
709,3,male,NaN,1,1,15.2458,C
439,2,male,31.0,0,0,10.5000,S
840,3,male,20.0,0,0,7.9250,S
720,2,female,6.0,0,1,33.0000,S
39,3,female,14.0,1,0,11.2417,C


In [10]:
X_train.shape,X_test.shape

((712, 7), (179, 7))

In [12]:
y_train.shape,y_test.shape

((712,), (179,))

In [14]:
df.isnull().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

In [16]:
#applying imputation
si_age = SimpleImputer() #two different imputers with two different strategey
si_embarked = SimpleImputer(strategy='most_frequent')

X_train_age = si_age.fit_transform(X_train[['Age']])
X_train_embarked = si_embarked.fit_transform(X_train[['Embarked']])

X_test_age = si_age.transform(X_test[['Age']])
X_test_embarked = si_embarked.transform(X_test[['Embarked']])   #[['attribute']] to make 2d


In [20]:
#one hot encoding sex and embarked
ohe_sex = OneHotEncoder(sparse_output=False,handle_unknown='ignore')
ohe_embarked = OneHotEncoder(sparse_output=False,handle_unknown='ignore')

X_train_sex = ohe_sex.fit_transform(X_train[['Sex']])
X_train_embarked = ohe_embarked.fit_transform(X_train[['Embarked']])

X_test_sex = ohe_sex.transform(X_test[['Sex']])
X_test_embarked = ohe_embarked.transform(X_test[['Embarked']])

In [22]:
X_train_sex

array([[0., 1.],
       [0., 1.],
       [0., 1.],
       ...,
       [0., 1.],
       [1., 0.],
       [0., 1.]])

In [23]:
X_train_embarked

array([[0., 0., 1., 0.],
       [0., 0., 1., 0.],
       [0., 0., 1., 0.],
       ...,
       [0., 0., 1., 0.],
       [0., 0., 1., 0.],
       [0., 0., 1., 0.]])

In [24]:
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5000,S
733,2,male,23.0,0,0,13.0000,S
382,3,male,32.0,0,0,7.9250,S
704,3,male,26.0,1,0,7.8542,S
813,3,female,6.0,4,2,31.2750,S


In [26]:
X_train_rem = X_train.drop(columns=['Sex','Age','Embarked'])
X_test_rem = X_test.drop(columns=['Sex','Age','Embarked'])

In [27]:
X_train_transformed = np.concatenate((X_train_rem,X_train_age,X_train_embarked,X_train_sex),axis = 1)
X_test_transformed = np.concatenate((X_test_rem,X_test_age,X_test_embarked,X_test_sex),axis = 1)

In [28]:
X_train_transformed

array([[1., 0., 0., ..., 0., 0., 1.],
       [2., 0., 0., ..., 0., 0., 1.],
       [3., 0., 0., ..., 0., 0., 1.],
       ...,
       [3., 2., 0., ..., 0., 0., 1.],
       [1., 1., 2., ..., 0., 1., 0.],
       [1., 0., 1., ..., 0., 0., 1.]])

In [29]:
X_test_transformed

array([[3., 1., 1., ..., 0., 0., 1.],
       [2., 0., 0., ..., 0., 0., 1.],
       [3., 0., 0., ..., 0., 0., 1.],
       ...,
       [3., 1., 5., ..., 0., 1., 0.],
       [2., 0., 0., ..., 0., 1., 0.],
       [3., 1., 1., ..., 0., 1., 0.]])

In [31]:
clf = DecisionTreeClassifier()
clf.fit(X_train_transformed,y_train)

DecisionTreeClassifier()

In [32]:
y_pred = clf.predict(X_test_transformed)

In [33]:
y_pred

array([0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0,
       0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1,
       0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1,
       0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0,
       1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0,
       0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0,
       0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0,
       0, 1, 1], dtype=int64)

In [34]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.776536312849162

In [37]:
# for exporing and creating a model
import pickle
import os

In [38]:
os.makedirs("models", exist_ok=True)
pickle.dump(ohe_sex,open('models/ohe_sex.pkl','wb'))
pickle.dump(ohe_embarked,open('models/ohe_embarked.pkl','wb'))
pickle.dump(clf,open('models/clf.pkl','wb'))

## Using Sklearn Pipelines

![strategy for pipelines](./pipelines.png)

In [6]:
import pandas as pd
import numpy as np

In [7]:
df = pd.read_csv('train.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [8]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'],inplace = True) #removing unwanted columns

In [9]:
#step 1 Train test split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['Survived']),df['Survived'],test_size=0.2)

In [10]:
X_train,X_test

(     Pclass     Sex   Age  SibSp  Parch     Fare Embarked
 470       3    male   NaN      0      0   7.2500        S
 542       3  female  11.0      4      2  31.2750        S
 198       3  female   NaN      0      0   7.7500        Q
 731       3    male  11.0      0      0  18.7875        C
 312       2  female  26.0      1      1  26.0000        S
 ..      ...     ...   ...    ...    ...      ...      ...
 106       3  female  21.0      0      0   7.6500        S
 328       3  female  31.0      1      1  20.5250        S
 197       3    male  42.0      0      1   8.4042        S
 147       3  female   9.0      2      2  34.3750        S
 53        2  female  29.0      1      0  26.0000        S
 
 [712 rows x 7 columns],
      Pclass     Sex   Age  SibSp  Parch     Fare Embarked
 645       1    male  48.0      1      0  76.7292        C
 482       3    male  50.0      0      0   8.0500        S
 765       1  female  51.0      1      0  77.9583        S
 688       3    male  18.0   

In [11]:
y_train,y_test

(470    0
 542    0
 198    1
 731    0
 312    0
       ..
 106    1
 328    1
 197    0
 147    0
 53     1
 Name: Survived, Length: 712, dtype: int64,
 645    1
 482    0
 765    1
 688    0
 886    0
       ..
 694    0
 245    0
 223    0
 248    1
 40     0
 Name: Survived, Length: 179, dtype: int64)

In [13]:
#checking missing values
X_train.isnull().sum(),X_test.isnull().sum()

(Pclass        0
 Sex           0
 Age         133
 SibSp         0
 Parch         0
 Fare          0
 Embarked      2
 dtype: int64,
 Pclass       0
 Sex          0
 Age         44
 SibSp        0
 Parch        0
 Fare         0
 Embarked     0
 dtype: int64)

In [41]:
#imputation transfer for handling missing values
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
trf1 = ColumnTransformer([
    ('impute_age',SimpleImputer(strategy='mean'),[2]), #2 means age , using index value instead of name to make pipeline smooth as this will create a numpy array bot datafarme and numpy array doesnot have name
    ('impute_embarked',SimpleImputer(strategy='most_frequent'),[6])
],remainder='passthrough') #if not used passthrough , rest of the column would have been dropped

In [42]:
#one hot encoding
from sklearn.preprocessing import OneHotEncoder
trf2 = ColumnTransformer([
    ('ohe_sex_embarked',OneHotEncoder(sparse_output=False,handle_unknown='ignore'),[1,6]), #drop_first is not true because using decision tree(no multicolinearity problem)
],remainder = 'passthrough')

In [43]:
#scaling
from sklearn.preprocessing import MinMaxScaler
trf3 = ColumnTransformer([
    ('scale',MinMaxScaler(),slice(0,10))  #10 cause after ohe extra columns 
])

In [44]:
#feature_selection ---> not necessary here , just to learn for future reference
from sklearn.feature_selection import SelectKBest,chi2
trf4 = SelectKBest(score_func=chi2,k=5)

In [45]:
#training the model using decision tree
from sklearn.tree import DecisionTreeClassifier
trf5 = DecisionTreeClassifier()

**Individual chains are now created now we have to join them using pipeline**

## create pipeline

In [46]:
from sklearn.pipeline import Pipeline
pipe = Pipeline([
    ('trf1',trf1),
    ('trf2',trf2),
    ('trf3',trf3),
    ('trf4',trf4),
    ('trf5',trf5)
])

## Pipeline vs make_pipeline

- pipeline requires naming of steps, make_pipeline does not
- (Same applies to ColumnTransformer vs make_column_transformer)

In [30]:
#alternative syntax
from sklearn.pipeline import make_pipeline
#pipe = make_pipeline(trf1,trf2,trf3,trf4,trf5)

In [47]:
from sklearn import set_config
set_config(display='diagram')

In [48]:
# train model
pipe.fit(X_train,y_train)

Pipeline(steps=[('trf1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_age', SimpleImputer(),
                                                  [2]),
                                                 ('impute_embarked',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [6])])),
                ('trf2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_sex_embarked',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [1, 6])])),
                ('trf3',
                 ColumnTransformer(transformers=[('scale', MinMaxScaler(),
                                                  slice(0, 10, None))])),
                ('trf4',
                 SelectKBest(k=5,
                             score_func=<function chi2 at 0x000001E3D74B3560>)),
                ('trf5', DecisionTreeClassifier())])

## Exploring pipeline

In [57]:
pipe.named_steps['trf1'].transformers_[0][1].statistics_ #mean is 29.3236.....

array([29.32369603])

In [58]:
pipe.named_steps['trf1'].transformers_[1][1].statistics_ #it slect southhamtion i.e S

array(['S'], dtype=object)

In [60]:
#predict
y_pred = pipe.predict(X_test)

In [61]:
y_pred

array([1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0,
       1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,
       0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1,
       0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0,
       0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0,
       0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 0], dtype=int64)

In [62]:
from sklearn.metrics import accuracy_score
accuracy_score(y_pred,y_test)

0.6536312849162011

**low score than above due to feature selection**, feature selection is implemented here just to show how to add it in the pipeline

In [63]:
# export 
import pickle
pickle.dump(pipe,open('pipe.pkl','wb'))